# IMC Prosperity Round 5 - Deep Quantitative Analysis
---
**50 products** across **10 categories** | Days 2, 3, 4 | ~1.5M price rows + ~35K trades

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (18, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')

DATA = 'ROUND_5/'

# Load all prices
prices = pd.concat([
    pd.read_csv(f'{DATA}prices_round_5_day_{d}.csv', sep=';') for d in [2,3,4]
], ignore_index=True)

# Load all trades
trades = pd.concat([
    pd.read_csv(f'{DATA}trades_round_5_day_{d}.csv', sep=';').assign(day=d) for d in [2,3,4]
], ignore_index=True)

# Create global timestamp
prices['global_ts'] = (prices['day'] - 2) * 1_000_000 + prices['timestamp']
trades['global_ts'] = (trades['day'] - 2) * 1_000_000 + trades['timestamp']

# Product categories
def get_category(p):
    for cat in ['GALAXY_SOUNDS', 'MICROCHIP', 'OXYGEN_SHAKE', 'PANEL', 'PEBBLES',
                'ROBOT', 'SLEEP_POD', 'SNACKPACK', 'TRANSLATOR', 'UV_VISOR']:
        if p.startswith(cat):
            return cat
    return 'OTHER'

prices['category'] = prices['product'].apply(get_category)
trades['category'] = trades['symbol'].apply(get_category)

# Spread
prices['spread'] = prices['ask_price_1'] - prices['bid_price_1']
prices['spread_pct'] = prices['spread'] / prices['mid_price'] * 100

products = sorted(prices['product'].unique())
categories = sorted(prices['category'].unique())

print(f'Prices: {len(prices):,} rows')
print(f'Trades: {len(trades):,} rows')
print(f'Products: {len(products)}')
print(f'Categories: {categories}')
print(f'\nProducts per category:')
for cat in categories:
    prods = [p for p in products if p.startswith(cat)]
    print(f'  {cat}: {len(prods)} -> {[p.replace(cat+"_","") for p in prods]}')

## 1. Price Overview - All Products Across All Days

In [ ]:
# Summary stats per product
summary = prices.groupby('product').agg(
    category=('category', 'first'),
    start_price=('mid_price', 'first'),
    end_price=('mid_price', 'last'),
    min_price=('mid_price', 'min'),
    max_price=('mid_price', 'max'),
    mean_price=('mid_price', 'mean'),
    std_price=('mid_price', 'std'),
    avg_spread=('spread', 'mean'),
    avg_spread_pct=('spread_pct', 'mean'),
).reset_index()

summary['total_return_pct'] = (summary['end_price'] / summary['start_price'] - 1) * 100
summary['range_pct'] = (summary['max_price'] - summary['min_price']) / summary['mean_price'] * 100
summary['volatility'] = summary['std_price'] / summary['mean_price'] * 100

summary = summary.sort_values('total_return_pct', ascending=False)
print(summary[['product','category','start_price','end_price','total_return_pct','volatility','avg_spread_pct','range_pct']].to_string(index=False))

In [ ]:
# Big overview: total return by product, colored by category
fig, ax = plt.subplots(figsize=(20, 10))
colors = plt.cm.tab10(np.linspace(0, 1, len(categories)))
cat_colors = dict(zip(categories, colors))

bars = ax.barh(summary['product'], summary['total_return_pct'],
               color=[cat_colors[c] for c in summary['category']])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Return (%)')
ax.set_title('Total Return (%) per Product - Day 2 to Day 4')
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cat_colors[c], label=c) for c in categories]
ax.legend(handles=legend_elements, loc='lower right', ncol=2)
plt.tight_layout()
plt.show()

## 2. Mid-Price Time Series by Category

In [ ]:
# Plot mid_price for all products, grouped by category
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    for prod in cat_products:
        pdf = prices[prices['product'] == prod]
        ax.plot(pdf['global_ts'], pdf['mid_price'], label=prod.replace(cat+'_',''), alpha=0.8, linewidth=0.7)
    ax.set_title(f'{cat} ({len(cat_products)} products)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Global Timestamp')
    ax.set_ylabel('Mid Price')
    ax.legend(fontsize=8, loc='best')
    # Add day boundaries
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4, linewidth=1)

plt.suptitle('Mid-Price Time Series by Category (Red dashed = day boundaries)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 3. Normalized Price (Index = 100) by Category - Relative Performance

In [ ]:
# Normalized prices for relative performance comparison
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].copy()
        start_p = pdf['mid_price'].iloc[0]
        pdf['normalized'] = pdf['mid_price'] / start_p * 100
        ax.plot(pdf['global_ts'], pdf['normalized'], label=prod.replace(cat+'_',''), alpha=0.8, linewidth=0.7)
    ax.axhline(100, color='black', linestyle='-', alpha=0.3)
    ax.set_title(f'{cat} - Normalized (base=100)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Global Timestamp')
    ax.set_ylabel('Normalized Price')
    ax.legend(fontsize=8, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Normalized Prices by Category (base=100 at start)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 4. Spread Analysis

In [ ]:
# Average spread by product
spread_summary = prices.groupby(['product','category']).agg(
    avg_spread=('spread', 'mean'),
    median_spread=('spread', 'median'),
    avg_spread_pct=('spread_pct', 'mean'),
    max_spread=('spread', 'max'),
    min_spread=('spread', 'min'),
    std_spread=('spread', 'std'),
).reset_index().sort_values('avg_spread_pct', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 12))

# Absolute spread
s = spread_summary.sort_values('avg_spread', ascending=True)
ax1.barh(s['product'], s['avg_spread'], color=[cat_colors[c] for c in s['category']])
ax1.set_xlabel('Average Spread (absolute)')
ax1.set_title('Average Bid-Ask Spread by Product')

# Percentage spread
s = spread_summary.sort_values('avg_spread_pct', ascending=True)
ax2.barh(s['product'], s['avg_spread_pct'], color=[cat_colors[c] for c in s['category']])
ax2.set_xlabel('Average Spread (%)')
ax2.set_title('Average Bid-Ask Spread (%) by Product')

plt.tight_layout()
plt.show()

print('\nSpread summary:')
print(spread_summary[['product','category','avg_spread','avg_spread_pct','max_spread']].to_string(index=False))

In [ ]:
# Spread over time by category
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    for prod in cat_products:
        pdf = prices[prices['product'] == prod]
        # Downsample for plot performance
        pdf_ds = pdf.iloc[::50]
        ax.plot(pdf_ds['global_ts'], pdf_ds['spread_pct'], label=prod.replace(cat+'_',''), alpha=0.7, linewidth=0.6)
    ax.set_title(f'{cat} - Spread (%)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Global Timestamp')
    ax.set_ylabel('Spread %')
    ax.legend(fontsize=8, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Bid-Ask Spread (%) Over Time by Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 5. Volatility Analysis - Rolling Standard Deviation

In [ ]:
# Compute returns and rolling volatility per product
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

vol_data = []

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].copy()
        pdf['ret'] = pdf['mid_price'].pct_change()
        pdf['rolling_vol'] = pdf['ret'].rolling(100).std() * 100  # as %
        
        vol_data.append({
            'product': prod, 'category': cat,
            'mean_vol': pdf['ret'].std() * 100,
            'max_ret': pdf['ret'].max() * 100,
            'min_ret': pdf['ret'].min() * 100,
        })
        
        pdf_ds = pdf.iloc[::50]
        ax.plot(pdf_ds['global_ts'], pdf_ds['rolling_vol'], label=prod.replace(cat+'_',''), alpha=0.7, linewidth=0.6)
    
    ax.set_title(f'{cat} - Rolling Volatility (100-tick)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Volatility (%)')
    ax.legend(fontsize=8, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Rolling Volatility (100-tick window) by Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

vol_df = pd.DataFrame(vol_data).sort_values('mean_vol', ascending=False)
print('\nVolatility Ranking:')
print(vol_df.to_string(index=False))

## 6. Intra-Category Correlation Analysis

In [ ]:
# Correlation of mid-prices within each category
fig, axes = plt.subplots(5, 2, figsize=(22, 35))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    # Pivot to get returns for each product
    pivot = prices[prices['category'] == cat].pivot_table(
        index='global_ts', columns='product', values='mid_price'
    )
    returns = pivot.pct_change().dropna()
    corr = returns.corr()
    
    # Clean labels
    labels = [c.replace(cat+'_','') for c in corr.columns]
    
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                xticklabels=labels, yticklabels=labels, ax=ax, vmin=-1, vmax=1,
                annot_kws={'size': 9})
    ax.set_title(f'{cat} - Return Correlations', fontsize=13, fontweight='bold')

plt.suptitle('Intra-Category Return Correlations', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Cross-Category Correlation Matrix (Category Averages)

In [ ]:
# Average mid-price return per category, then correlate
cat_avg = prices.groupby(['global_ts', 'category'])['mid_price'].mean().unstack()
cat_returns = cat_avg.pct_change().dropna()
cat_corr = cat_returns.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(cat_corr, dtype=bool), k=1)
sns.heatmap(cat_corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, vmin=-1, vmax=1, annot_kws={'size': 12})
ax.set_title('Cross-Category Return Correlation', fontsize=15)
plt.tight_layout()
plt.show()

## 8. Trade Analysis - External Trades (XIRECS)

In [ ]:
# Trades overview
print('=== TRADE DATA STRUCTURE ===')
print(trades.head(10))
print(f'\nTotal trades: {len(trades):,}')
print(f'\nCurrency values: {trades["currency"].unique()}')
print(f'\nBuyer values: {trades["buyer"].unique()}')
print(f'\nSeller values: {trades["seller"].unique()}')
print(f'\nQuantity distribution:')
print(trades['quantity'].describe())
print(f'\nQuantity by day:')
print(trades.groupby('day')['quantity'].describe())

# Trades per product
trade_counts = trades.groupby('symbol').agg(
    num_trades=('price','count'),
    total_volume=('quantity','sum'),
    avg_price=('price','mean'),
    avg_quantity=('quantity','mean'),
).sort_values('num_trades', ascending=False)
print('\n=== TRADES PER PRODUCT ===')
print(trade_counts.to_string())

In [ ]:
# Trade timestamps - when do trades happen?
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for i, day in enumerate([2, 3, 4]):
    ax = axes[i]
    day_trades = trades[trades['day'] == day]
    ax.hist(day_trades['timestamp'], bins=100, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f'Day {day} - Trade Timestamp Distribution ({len(day_trades)} trades)', fontsize=13)
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Number of trades')

plt.suptitle('When Do Trades Happen?', fontsize=16)
plt.tight_layout()
plt.show()

# Unique timestamps in trades
for day in [2, 3, 4]:
    dt = trades[trades['day'] == day]
    ts = sorted(dt['timestamp'].unique())
    print(f'Day {day}: {len(ts)} unique timestamps, range [{ts[0]}, {ts[-1]}]')
    if len(ts) > 1:
        diffs = np.diff(ts)
        print(f'  Interval: min={diffs.min()}, max={diffs.max()}, median={np.median(diffs)}, mean={diffs.mean():.0f}')

In [ ]:
# Trade price vs mid price - are trades at fair value?
# Merge trades with closest mid_price
trade_vs_mid = []

for day in [2, 3, 4]:
    day_prices = prices[prices['day'] == day]
    day_trades = trades[trades['day'] == day]
    
    for _, trade in day_trades.iterrows():
        # Find closest price timestamp for this product
        prod_prices = day_prices[day_prices['product'] == trade['symbol']]
        if len(prod_prices) == 0:
            continue
        idx = (prod_prices['timestamp'] - trade['timestamp']).abs().idxmin()
        mid = prod_prices.loc[idx, 'mid_price']
        bid = prod_prices.loc[idx, 'bid_price_1']
        ask = prod_prices.loc[idx, 'ask_price_1']
        
        trade_vs_mid.append({
            'symbol': trade['symbol'],
            'day': day,
            'trade_price': trade['price'],
            'mid_price': mid,
            'bid_price': bid,
            'ask_price': ask,
            'deviation': trade['price'] - mid,
            'deviation_pct': (trade['price'] - mid) / mid * 100,
            'category': get_category(trade['symbol']),
        })

tvm = pd.DataFrame(trade_vs_mid)
print('Trade Price vs Mid Price Deviation:')
print(tvm.groupby('category')['deviation_pct'].describe().round(3))

In [ ]:
# Distribution of trade price deviations from mid
fig, axes = plt.subplots(5, 2, figsize=(20, 25))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_data = tvm[tvm['category'] == cat]
    if len(cat_data) > 0:
        cat_data['deviation_pct'].hist(bins=50, ax=ax, alpha=0.7, edgecolor='black', linewidth=0.3)
        ax.axvline(0, color='red', linewidth=1)
        ax.set_title(f'{cat} - Trade Deviation from Mid (%) [n={len(cat_data)}]', fontweight='bold')
        ax.set_xlabel('Deviation %')
        mean_dev = cat_data['deviation_pct'].mean()
        ax.axvline(mean_dev, color='green', linestyle='--', label=f'Mean: {mean_dev:.3f}%')
        ax.legend()

plt.suptitle('Trade Price Deviation from Mid-Price (%)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 9. Category-Level Analysis: Mean Reversion & Momentum

In [ ]:
# Category basket price (equal-weighted average)
cat_basket = prices.groupby(['global_ts', 'category'])['mid_price'].mean().reset_index()

fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    basket = cat_basket[cat_basket['category'] == cat]
    
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].copy()
        merged = pdf.merge(basket[['global_ts','mid_price']], on='global_ts', suffixes=('','_basket'))
        merged['relative'] = merged['mid_price'] / merged['mid_price_basket']
        merged_ds = merged.iloc[::50]
        ax.plot(merged_ds['global_ts'], merged_ds['relative'], label=prod.replace(cat+'_',''), alpha=0.8, linewidth=0.7)
    
    ax.set_title(f'{cat} - Price / Category Basket', fontsize=13, fontweight='bold')
    ax.set_ylabel('Relative to Basket')
    ax.legend(fontsize=8, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Individual Product Price / Category Average Basket', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Z-score of each product vs its category mean (potential mean-reversion signal)
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    # Get returns
    pivot = prices[prices['category'] == cat].pivot_table(
        index='global_ts', columns='product', values='mid_price'
    )
    returns = pivot.pct_change().dropna()
    
    # Z-score of cumulative return vs category mean
    cum_ret = (1 + returns).cumprod()
    cat_mean = cum_ret.mean(axis=1)
    
    for prod in cat_products:
        if prod in cum_ret.columns:
            z = (cum_ret[prod] - cat_mean) / cum_ret.std(axis=1)
            z_ds = z.iloc[::50]
            ax.plot(z_ds.index, z_ds.values, label=prod.replace(cat+'_',''), alpha=0.7, linewidth=0.6)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline(2, color='red', linestyle='--', alpha=0.5)
    ax.axhline(-2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{cat} - Z-Score vs Category Mean', fontsize=13, fontweight='bold')
    ax.set_ylabel('Z-Score')
    ax.legend(fontsize=8, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Cumulative Return Z-Score vs Category Mean (|Z|>2 = potential trade)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 10. Pairs Analysis - Spread Between Pairs Within Categories

In [ ]:
# For each category, compute pairwise price spread ratios
from itertools import combinations

# Focus on categories with strong internal structure
fig, axes = plt.subplots(5, 2, figsize=(22, 30))
axes = axes.flatten()

pair_stats = []

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    pivot = prices[prices['category'] == cat].pivot_table(
        index='global_ts', columns='product', values='mid_price'
    )
    
    # Show top pairs by mean-reversion potential (low std of ratio)
    pairs = list(combinations(cat_products, 2))
    
    for p1, p2 in pairs[:6]:  # Top 6 pairs per category
        if p1 in pivot.columns and p2 in pivot.columns:
            ratio = pivot[p1] / pivot[p2]
            ratio_ds = ratio.iloc[::100]
            short_name = f"{p1.replace(cat+'_','')}/{p2.replace(cat+'_','')}"
            ax.plot(ratio_ds.index, ratio_ds.values, label=short_name, alpha=0.7, linewidth=0.6)
            
            pair_stats.append({
                'category': cat, 'pair': f'{p1}/{p2}',
                'mean_ratio': ratio.mean(), 'std_ratio': ratio.std(),
                'cv': ratio.std() / ratio.mean(),
            })
    
    ax.set_title(f'{cat} - Pair Ratios', fontsize=13, fontweight='bold')
    ax.set_ylabel('Price Ratio')
    ax.legend(fontsize=7, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Pairwise Price Ratios Within Categories', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

pair_df = pd.DataFrame(pair_stats).sort_values('cv')
print('\nMost Stable Pairs (lowest CV = best mean reversion candidates):')
print(pair_df.head(20).to_string(index=False))

## 11. Order Book Depth Analysis

In [ ]:
# Order book depth - total volume at each level
depth_cols = ['bid_volume_1','bid_volume_2','bid_volume_3','ask_volume_1','ask_volume_2','ask_volume_3']
prices['total_bid_vol'] = prices[['bid_volume_1','bid_volume_2','bid_volume_3']].sum(axis=1)
prices['total_ask_vol'] = prices[['ask_volume_1','ask_volume_2','ask_volume_3']].sum(axis=1)
prices['vol_imbalance'] = (prices['total_bid_vol'] - prices['total_ask_vol']) / (prices['total_bid_vol'] + prices['total_ask_vol'])

# Average depth by category
depth_by_cat = prices.groupby('category').agg(
    avg_bid_vol_1=('bid_volume_1', 'mean'),
    avg_ask_vol_1=('ask_volume_1', 'mean'),
    avg_total_bid=('total_bid_vol', 'mean'),
    avg_total_ask=('total_ask_vol', 'mean'),
    avg_imbalance=('vol_imbalance', 'mean'),
).round(2)

print('=== ORDER BOOK DEPTH BY CATEGORY ===')
print(depth_by_cat)

# Depth by product
depth_by_prod = prices.groupby('product').agg(
    category=('category','first'),
    avg_bid_vol_1=('bid_volume_1', 'mean'),
    avg_ask_vol_1=('ask_volume_1', 'mean'),
    avg_total_bid=('total_bid_vol', 'mean'),
    avg_total_ask=('total_ask_vol', 'mean'),
    avg_imbalance=('vol_imbalance', 'mean'),
    num_levels_bid=('bid_volume_3', lambda x: (x > 0).mean()),
).round(3)

print('\n=== ORDER BOOK DEPTH BY PRODUCT ===')
print(depth_by_prod.sort_values('avg_total_bid', ascending=False).to_string())

In [ ]:
# Volume imbalance over time by category
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].iloc[::100]
        ax.plot(pdf['global_ts'], pdf['vol_imbalance'], label=prod.replace(cat+'_',''), alpha=0.6, linewidth=0.5)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{cat} - Volume Imbalance', fontsize=13, fontweight='bold')
    ax.set_ylabel('Imbalance (Bid-Ask)/(Bid+Ask)')
    ax.legend(fontsize=7, loc='best')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Order Book Volume Imbalance Over Time', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 12. Day-over-Day Price Changes (Overnight Gaps)

In [ ]:
# Day boundaries - last price day N vs first price day N+1
overnight = []
for prod in products:
    for d1, d2 in [(2,3), (3,4)]:
        p1 = prices[(prices['product'] == prod) & (prices['day'] == d1)]
        p2 = prices[(prices['product'] == prod) & (prices['day'] == d2)]
        if len(p1) > 0 and len(p2) > 0:
            close = p1['mid_price'].iloc[-1]
            open_ = p2['mid_price'].iloc[0]
            overnight.append({
                'product': prod, 'category': get_category(prod),
                'transition': f'Day {d1}->Day {d2}',
                'close': close, 'open': open_,
                'gap': open_ - close,
                'gap_pct': (open_ - close) / close * 100,
            })

on_df = pd.DataFrame(overnight)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 12))

# Day 2->3
d23 = on_df[on_df['transition'] == 'Day 2->Day 3'].sort_values('gap_pct')
ax1.barh(d23['product'], d23['gap_pct'], color=[cat_colors[get_category(p)] for p in d23['product']])
ax1.axvline(0, color='black')
ax1.set_xlabel('Gap %')
ax1.set_title('Overnight Gap: Day 2 Close -> Day 3 Open (%)')

# Day 3->4
d34 = on_df[on_df['transition'] == 'Day 3->Day 4'].sort_values('gap_pct')
ax2.barh(d34['product'], d34['gap_pct'], color=[cat_colors[get_category(p)] for p in d34['product']])
ax2.axvline(0, color='black')
ax2.set_xlabel('Gap %')
ax2.set_title('Overnight Gap: Day 3 Close -> Day 4 Open (%)')

plt.tight_layout()
plt.show()

print('\nOvernight Gap Summary by Category:')
print(on_df.groupby(['category','transition'])['gap_pct'].agg(['mean','std','min','max']).round(2).to_string())

## 13. Autocorrelation Analysis (Predictability)

In [ ]:
# Return autocorrelation at various lags
lags = [1, 5, 10, 20, 50, 100]
autocorr_data = []

for prod in products:
    pdf = prices[prices['product'] == prod].copy()
    pdf['ret'] = pdf['mid_price'].pct_change()
    for lag in lags:
        ac = pdf['ret'].autocorr(lag=lag)
        autocorr_data.append({
            'product': prod, 'category': get_category(prod),
            'lag': lag, 'autocorr': ac
        })

ac_df = pd.DataFrame(autocorr_data)

# Plot autocorrelation heatmap per category
fig, axes = plt.subplots(5, 2, figsize=(20, 30))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_ac = ac_df[ac_df['category'] == cat]
    pivot = cat_ac.pivot_table(index='product', columns='lag', values='autocorr')
    labels = [p.replace(cat+'_','') for p in pivot.index]
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                yticklabels=labels, ax=ax, vmin=-0.3, vmax=0.3)
    ax.set_title(f'{cat} - Return Autocorrelation', fontweight='bold')

plt.suptitle('Return Autocorrelation at Various Lags', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# Most predictable products
lag1 = ac_df[ac_df['lag'] == 1].copy()
lag1['abs_ac'] = lag1['autocorr'].abs()
print('\nStrongest Lag-1 Autocorrelation (most predictable):')
print(lag1.sort_values('abs_ac', ascending=False).head(20)[['product','autocorr','abs_ac']].to_string(index=False))

## 14. Intraday Patterns - Average Price Movement by Timestamp

In [ ]:
# Average return by timestamp bucket (intraday patterns)
prices['ts_bucket'] = (prices['timestamp'] // 10000) * 10000  # 10k tick buckets

fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].copy()
        pdf['ret'] = pdf['mid_price'].pct_change()
        intraday = pdf.groupby('ts_bucket')['ret'].mean() * 100
        ax.plot(intraday.index, intraday.values, label=prod.replace(cat+'_',''), alpha=0.7, linewidth=0.8)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{cat} - Avg Return by Intraday Bucket', fontsize=13, fontweight='bold')
    ax.set_ylabel('Avg Return (%)')
    ax.legend(fontsize=7, loc='best')

plt.suptitle('Intraday Average Return Patterns (10K-tick buckets)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 15. Trade Impact Analysis - Price Movement Around Trades

In [ ]:
# For each category: overlay trade times on price charts
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    # Pick one representative product per category
    prod = cat_products[0]
    pdf = prices[prices['product'] == prod]
    tdf = trades[trades['symbol'] == prod]
    
    ax.plot(pdf['global_ts'], pdf['mid_price'], alpha=0.7, linewidth=0.5, label='Mid Price')
    ax.scatter(tdf['global_ts'], tdf['price'], color='red', s=15, alpha=0.7, zorder=5, label='Trades')
    ax.set_title(f'{prod} - Prices + Trade Points', fontsize=12, fontweight='bold')
    ax.set_ylabel('Price')
    ax.legend(fontsize=9)
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='green', linestyle='--', alpha=0.4)

plt.suptitle('Price + External Trade Overlay (one product per category)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 16. Category Basket Returns & Dispersion

In [ ]:
# Category-level dispersion (spread of returns within category)
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    pivot = prices[prices['category'] == cat].pivot_table(
        index='global_ts', columns='product', values='mid_price'
    )
    
    # Normalize
    norm = pivot / pivot.iloc[0] * 100
    
    # Compute cross-sectional dispersion
    dispersion = norm.std(axis=1)
    disp_ds = dispersion.iloc[::100]
    ax.plot(disp_ds.index, disp_ds.values, color='purple', linewidth=1)
    ax.fill_between(disp_ds.index, 0, disp_ds.values, alpha=0.3, color='purple')
    ax.set_title(f'{cat} - Cross-Sectional Dispersion', fontsize=13, fontweight='bold')
    ax.set_ylabel('Std of Normalized Prices')
    for d in [1_000_000, 2_000_000]:
        ax.axvline(d, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Intra-Category Price Dispersion (Higher = More Divergence = More Opportunity)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 17. PnL Analysis

In [ ]:
# PnL from the prices data
pnl_data = prices.groupby(['global_ts','product','category'])['profit_and_loss'].first().reset_index()

# Check if PnL changes
pnl_summary = pnl_data.groupby('product')['profit_and_loss'].agg(['min','max','mean','std','last'])
print('PnL Summary (checking if PnL varies):')
print(pnl_summary[pnl_summary['std'] > 0].to_string() if len(pnl_summary[pnl_summary['std'] > 0]) > 0 else 'All PnL are constant (0.0) - no trading PnL tracked in this data')

## 18. Product Ranking Dashboard - Strategy-Ready Summary

In [ ]:
# Comprehensive ranking for strategy selection
ranking = summary[['product','category','total_return_pct','volatility','avg_spread_pct','range_pct']].copy()

# Add autocorrelation
lag1_map = lag1.set_index('product')['autocorr'].to_dict()
ranking['autocorr_lag1'] = ranking['product'].map(lag1_map)

# Add depth
depth_map = depth_by_prod['avg_total_bid'].to_dict()
ranking['avg_depth'] = ranking['product'].map(depth_map)

# Score for market-making potential (tight spread, high depth, low volatility)
ranking['mm_score'] = (
    (1 - ranking['avg_spread_pct'].rank(pct=True)) * 0.4 +  # tighter spread = better
    ranking['avg_depth'].rank(pct=True) * 0.3 +  # more depth = better
    (1 - ranking['volatility'].rank(pct=True)) * 0.3  # lower vol = better for MM
)

# Score for momentum/trend potential (high return, high vol, negative autocorr)
ranking['trend_score'] = (
    ranking['range_pct'].rank(pct=True) * 0.4 +
    ranking['volatility'].rank(pct=True) * 0.3 +
    ranking['total_return_pct'].abs().rank(pct=True) * 0.3
)

print('=== TOP MARKET-MAKING CANDIDATES ===')
print(ranking.sort_values('mm_score', ascending=False).head(15)[['product','category','avg_spread_pct','volatility','avg_depth','mm_score']].to_string(index=False))

print('\n=== TOP TREND/MOMENTUM CANDIDATES ===')
print(ranking.sort_values('trend_score', ascending=False).head(15)[['product','category','total_return_pct','range_pct','volatility','trend_score']].to_string(index=False))

In [ ]:
# Visual ranking dashboard
fig, axes = plt.subplots(2, 3, figsize=(22, 14))

# 1. Return vs Volatility
ax = axes[0, 0]
for cat in categories:
    mask = ranking['category'] == cat
    ax.scatter(ranking[mask]['volatility'], ranking[mask]['total_return_pct'],
               label=cat, color=cat_colors[cat], s=40, alpha=0.8)
ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Total Return (%)')
ax.set_title('Return vs Volatility')
ax.axhline(0, color='black', linewidth=0.5)
ax.legend(fontsize=7, ncol=2)

# 2. Spread vs Volatility
ax = axes[0, 1]
for cat in categories:
    mask = ranking['category'] == cat
    ax.scatter(ranking[mask]['avg_spread_pct'], ranking[mask]['volatility'],
               label=cat, color=cat_colors[cat], s=40, alpha=0.8)
ax.set_xlabel('Avg Spread (%)')
ax.set_ylabel('Volatility (%)')
ax.set_title('Spread vs Volatility')
ax.legend(fontsize=7, ncol=2)

# 3. MM Score vs Trend Score
ax = axes[0, 2]
for cat in categories:
    mask = ranking['category'] == cat
    ax.scatter(ranking[mask]['mm_score'], ranking[mask]['trend_score'],
               label=cat, color=cat_colors[cat], s=40, alpha=0.8)
ax.set_xlabel('Market-Making Score')
ax.set_ylabel('Trend Score')
ax.set_title('MM Score vs Trend Score')
ax.legend(fontsize=7, ncol=2)

# 4. Top MM candidates bar
ax = axes[1, 0]
top_mm = ranking.sort_values('mm_score', ascending=True).tail(15)
ax.barh(top_mm['product'], top_mm['mm_score'], color=[cat_colors[c] for c in top_mm['category']])
ax.set_title('Top 15 Market-Making Scores')
ax.set_xlabel('MM Score')

# 5. Top Trend candidates bar
ax = axes[1, 1]
top_tr = ranking.sort_values('trend_score', ascending=True).tail(15)
ax.barh(top_tr['product'], top_tr['trend_score'], color=[cat_colors[c] for c in top_tr['category']])
ax.set_title('Top 15 Trend Scores')
ax.set_xlabel('Trend Score')

# 6. Autocorrelation distribution
ax = axes[1, 2]
for cat in categories:
    mask = ranking['category'] == cat
    ax.scatter(ranking[mask]['autocorr_lag1'], ranking[mask]['volatility'],
               label=cat, color=cat_colors[cat], s=40, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag-1 Autocorrelation')
ax.set_ylabel('Volatility (%)')
ax.set_title('Autocorrelation vs Volatility')
ax.legend(fontsize=7, ncol=2)

plt.suptitle('Strategy Selection Dashboard', fontsize=16)
plt.tight_layout()
plt.show()

## 19. Trade Timing Analysis - Per Category

In [ ]:
# When do external trades happen for each category?
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_trades = trades[trades['category'] == cat]
    
    # Timestamps
    unique_ts = sorted(cat_trades['timestamp'].unique())
    trades_per_ts = cat_trades.groupby('timestamp').size()
    
    ax.bar(trades_per_ts.index, trades_per_ts.values, width=200, alpha=0.7)
    ax.set_title(f'{cat} - Trade Count by Timestamp ({len(unique_ts)} unique times)', fontweight='bold')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('# Trades')

plt.suptitle('Trade Frequency Distribution by Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# Trade frequency per category
print('\nTrade count by category and day:')
print(trades.groupby(['category','day']).size().unstack(fill_value=0))

## 20. Return Distribution Analysis (Tail Risk)

In [ ]:
# Return distributions per category
fig, axes = plt.subplots(5, 2, figsize=(22, 28))
axes = axes.flatten()

dist_stats = []

for idx, cat in enumerate(categories):
    ax = axes[idx]
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    for prod in cat_products:
        pdf = prices[prices['product'] == prod].copy()
        pdf['ret'] = pdf['mid_price'].pct_change().dropna()
        rets = pdf['ret'].dropna()
        
        ax.hist(rets * 100, bins=100, alpha=0.4, label=prod.replace(cat+'_',''), density=True)
        
        from scipy import stats as sp_stats
        dist_stats.append({
            'product': prod, 'category': cat,
            'mean_ret': rets.mean() * 100,
            'std_ret': rets.std() * 100,
            'skew': rets.skew(),
            'kurtosis': rets.kurtosis(),
            'pct_1': np.percentile(rets*100, 1),
            'pct_99': np.percentile(rets*100, 99),
        })
    
    ax.set_title(f'{cat} - Return Distribution', fontweight='bold')
    ax.set_xlabel('Return (%)')
    ax.legend(fontsize=7, loc='best')

plt.suptitle('Return Distributions by Category', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

dist_df = pd.DataFrame(dist_stats).sort_values('kurtosis', ascending=False)
print('\nReturn Distribution Stats (sorted by kurtosis - fat tails):')
print(dist_df.to_string(index=False))

## 21. Trade Price Prediction - Can External Trade Prices Predict Future Mid Price?

In [ ]:
# Check: does deviation of trade price from mid predict future mid price movement?
prediction_data = []

for day in [2, 3, 4]:
    day_prices = prices[prices['day'] == day]
    day_trades = trades[trades['day'] == day]
    
    for _, trade in day_trades.iterrows():
        prod_prices = day_prices[day_prices['product'] == trade['symbol']].copy()
        if len(prod_prices) < 10:
            continue
        
        # Find current mid
        mask_before = prod_prices['timestamp'] <= trade['timestamp']
        if mask_before.sum() == 0:
            continue
        current_mid = prod_prices[mask_before]['mid_price'].iloc[-1]
        
        # Find mid 100 ticks later
        mask_after = prod_prices['timestamp'] > trade['timestamp']
        future_prices = prod_prices[mask_after]
        if len(future_prices) < 50:
            continue
        future_mid = future_prices['mid_price'].iloc[min(49, len(future_prices)-1)]
        
        deviation = (trade['price'] - current_mid) / current_mid * 100
        future_ret = (future_mid - current_mid) / current_mid * 100
        
        prediction_data.append({
            'symbol': trade['symbol'],
            'category': get_category(trade['symbol']),
            'deviation': deviation,
            'future_ret': future_ret,
        })

pred_df = pd.DataFrame(prediction_data)

if len(pred_df) > 0:
    fig, axes = plt.subplots(5, 2, figsize=(22, 28))
    axes = axes.flatten()
    
    for idx, cat in enumerate(categories):
        ax = axes[idx]
        cat_pred = pred_df[pred_df['category'] == cat]
        if len(cat_pred) > 5:
            ax.scatter(cat_pred['deviation'], cat_pred['future_ret'], alpha=0.3, s=10)
            # Fit line
            z = np.polyfit(cat_pred['deviation'], cat_pred['future_ret'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(cat_pred['deviation'].min(), cat_pred['deviation'].max(), 50)
            ax.plot(x_line, p(x_line), 'r-', linewidth=2)
            corr_val = cat_pred['deviation'].corr(cat_pred['future_ret'])
            ax.set_title(f'{cat} - Trade Dev vs Future Return (r={corr_val:.3f})', fontweight='bold')
        ax.set_xlabel('Trade Price Deviation from Mid (%)')
        ax.set_ylabel('Future Return (~50 ticks) (%)')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
    
    plt.suptitle('Trade Price Deviation vs Future Mid-Price Return', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()

## 22. Category Mean-Reversion Backtest Signal

In [ ]:
# Simple mean-reversion backtest: 
# If product is > 1 std below category mean -> buy signal
# If product is > 1 std above category mean -> sell signal
# Check if this generates positive returns

mr_results = []

for cat in categories:
    cat_products = sorted(prices[prices['category'] == cat]['product'].unique())
    
    pivot = prices[prices['category'] == cat].pivot_table(
        index='global_ts', columns='product', values='mid_price'
    )
    
    returns = pivot.pct_change().dropna()
    cum_ret = (1 + returns).cumprod()
    cat_mean = cum_ret.mean(axis=1)
    cat_std = cum_ret.std(axis=1)
    
    for prod in cat_products:
        if prod not in cum_ret.columns:
            continue
        z = (cum_ret[prod] - cat_mean) / cat_std
        
        # Forward return (10 ticks)
        fwd_ret = returns[prod].shift(-10)
        
        # Signal: buy when z < -1, sell when z > 1
        buy_mask = z < -1
        sell_mask = z > 1
        
        buy_ret = fwd_ret[buy_mask].mean() * 100 if buy_mask.sum() > 0 else np.nan
        sell_ret = fwd_ret[sell_mask].mean() * 100 if sell_mask.sum() > 0 else np.nan
        
        mr_results.append({
            'product': prod, 'category': cat,
            'buy_signals': buy_mask.sum(),
            'sell_signals': sell_mask.sum(),
            'avg_buy_ret_10tick': buy_ret,
            'avg_sell_ret_10tick': sell_ret,
            'combined_pnl': (buy_ret if not np.isnan(buy_ret) else 0) - (sell_ret if not np.isnan(sell_ret) else 0),
        })

mr_df = pd.DataFrame(mr_results).sort_values('combined_pnl', ascending=False)
print('=== MEAN REVERSION SIGNAL BACKTEST ===')
print('(Buy when Z < -1, Sell when Z > 1, measure 10-tick forward return)')
print()
print(mr_df.to_string(index=False))

# Summary by category
print('\n=== BY CATEGORY ===')
print(mr_df.groupby('category')['combined_pnl'].agg(['mean','sum']).sort_values('sum', ascending=False))

## 23. Price Level Analysis - Support/Resistance

In [ ]:
# Day-by-day OHLC for each product
ohlc = prices.groupby(['product','day','category']).agg(
    open=('mid_price', 'first'),
    high=('mid_price', 'max'),
    low=('mid_price', 'min'),
    close=('mid_price', 'last'),
    avg_volume=('bid_volume_1', 'mean'),
).reset_index()

ohlc['daily_range_pct'] = (ohlc['high'] - ohlc['low']) / ohlc['open'] * 100
ohlc['daily_return_pct'] = (ohlc['close'] / ohlc['open'] - 1) * 100

# Pivot for display
print('=== DAILY OHLC SUMMARY ===')
for day in [2, 3, 4]:
    print(f'\n--- DAY {day} ---')
    d = ohlc[ohlc['day'] == day].sort_values('daily_return_pct', ascending=False)
    print(d[['product','category','open','high','low','close','daily_return_pct','daily_range_pct']].to_string(index=False))

In [ ]:
# Day-over-day return pattern: does day 2->3 return predict day 3->4?
day_rets = ohlc.pivot_table(index='product', columns='day', values='daily_return_pct')
day_rets.columns = [f'day{d}_ret' for d in day_rets.columns]

if 'day2_ret' in day_rets.columns and 'day3_ret' in day_rets.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # Day 2 vs Day 3
    ax1.scatter(day_rets['day2_ret'], day_rets['day3_ret'], alpha=0.6, s=30)
    corr23 = day_rets['day2_ret'].corr(day_rets['day3_ret'])
    ax1.set_xlabel('Day 2 Return (%)')
    ax1.set_ylabel('Day 3 Return (%)')
    ax1.set_title(f'Day 2 vs Day 3 Returns (r={corr23:.3f})')
    ax1.axhline(0, color='black', linewidth=0.5)
    ax1.axvline(0, color='black', linewidth=0.5)
    
    # Day 3 vs Day 4
    if 'day4_ret' in day_rets.columns:
        ax2.scatter(day_rets['day3_ret'], day_rets['day4_ret'], alpha=0.6, s=30)
        corr34 = day_rets['day3_ret'].corr(day_rets['day4_ret'])
        ax2.set_xlabel('Day 3 Return (%)')
        ax2.set_ylabel('Day 4 Return (%)')
        ax2.set_title(f'Day 3 vs Day 4 Returns (r={corr34:.3f})')
        ax2.axhline(0, color='black', linewidth=0.5)
        ax2.axvline(0, color='black', linewidth=0.5)
    
    plt.suptitle('Daily Return Persistence (Momentum or Mean Reversion?)', fontsize=15)
    plt.tight_layout()
    plt.show()

## 24. Full Cross-Product Correlation Heatmap (All 50 Products)

In [ ]:
# Full 50x50 correlation matrix
full_pivot = prices.pivot_table(index='global_ts', columns='product', values='mid_price')
full_returns = full_pivot.pct_change().dropna()
full_corr = full_returns.corr()

fig, ax = plt.subplots(figsize=(24, 20))

# Sort by category for visual grouping
sorted_prods = sorted(full_corr.columns, key=lambda x: (get_category(x), x))
full_corr_sorted = full_corr.loc[sorted_prods, sorted_prods]

sns.heatmap(full_corr_sorted, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax,
            linewidths=0.1)
ax.set_title('Full 50x50 Product Return Correlation Matrix', fontsize=16)
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Most correlated pairs across categories
cross_cat_corr = []
for i in range(len(sorted_prods)):
    for j in range(i+1, len(sorted_prods)):
        p1, p2 = sorted_prods[i], sorted_prods[j]
        c1, c2 = get_category(p1), get_category(p2)
        if c1 != c2:
            cross_cat_corr.append({
                'product_1': p1, 'product_2': p2,
                'cat_1': c1, 'cat_2': c2,
                'correlation': full_corr_sorted.loc[p1, p2]
            })

cc_df = pd.DataFrame(cross_cat_corr).sort_values('correlation', key=abs, ascending=False)
print('\nTop Cross-Category Correlations:')
print(cc_df.head(20).to_string(index=False))
print('\nMost Negative Cross-Category Correlations:')
print(cc_df.sort_values('correlation').head(20).to_string(index=False))

## 25. Key Strategy Insights Summary

In [ ]:
print('=' * 80)
print('         IMC PROSPERITY ROUND 5 - STRATEGY INSIGHTS SUMMARY')
print('=' * 80)

print('\n1. DATA OVERVIEW')
print(f'   - 50 products in 10 categories')
print(f'   - 3 days of data (Day 2-4), ~500K price rows/day')
print(f'   - External trades (XIRECS) occur periodically (~35K total)')

print('\n2. PRODUCT CATEGORIES (5 products each):')
for cat in categories:
    prods = [p for p in products if p.startswith(cat)]
    avg_ret = summary[summary['category'] == cat]['total_return_pct'].mean()
    avg_vol = summary[summary['category'] == cat]['volatility'].mean()
    avg_sprd = summary[summary['category'] == cat]['avg_spread_pct'].mean()
    print(f'   {cat}: avg_ret={avg_ret:.1f}%, vol={avg_vol:.2f}%, spread={avg_sprd:.3f}%')

print('\n3. TOP MOVERS (Biggest total returns):')
top_movers = summary.head(10)
for _, row in top_movers.iterrows():
    print(f'   {row["product"]}: {row["total_return_pct"]:+.1f}%')

print('\n4. BOTTOM MOVERS:')
bottom_movers = summary.tail(10)
for _, row in bottom_movers.iterrows():
    print(f'   {row["product"]}: {row["total_return_pct"]:+.1f}%')

print('\n5. KEY OBSERVATIONS:')
print('   - All products start at 10,000 on Day 2 then diverge significantly')
print('   - Products within categories are correlated but NOT identical')
print('   - Spreads vary by category (SNACKPACK widest, ROBOT/MICROCHIP tightest)')
print('   - External trades happen at regular intervals per category')
print('   - Trade quantities decrease each day (4 -> 2 -> 1)')

print('\n6. STRATEGY OPPORTUNITIES:')
print('   a) MARKET MAKING: Products with tight spreads, high depth, low volatility')
print('   b) STAT ARB / PAIRS: Correlated products within categories with stable ratios')
print('   c) MEAN REVERSION: Products deviating from category mean tend to revert')
print('   d) TRADE SIGNAL: External trade price deviations may predict direction')
print('   e) MOMENTUM: Strong day-over-day return persistence in some products')

print('\n' + '=' * 80)